Developed by Francisco J. Lima, María Cuadrado, Fernando Gallego & Gloria Corpas, Lexytrad Research Group, University of Málaga.


### Imports


In [1]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from typing import Tuple, List
import re

### Global vars

In [2]:
SEED = 42
N_SPLITS = 5
OUT_DIR = "../data/kfold"

### Auxiliar functions

In [3]:
def build_parallel_dataframe(
    base_dir: str = "../data/FarmaCorp/FarmaCorp_par"
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build three dataframes from a parallel Spanish-English corpus.

    Args:
        base_dir: Path to the root directory containing subfolders 
                  with `es/` and `en/` text files.

    Returns:
        A tuple of three DataFrames:
        - parallel_df: aligned sentence pairs with corpus and filename metadata
        - reviewable_df: misaligned pairs with small differences (<= 3 lines)
        - non_reviewable_df: misaligned pairs with larger differences (> 3 lines)
    """
    records = []
    reviewable = []
    non_reviewable = []

    # Iterate through each subfolder (one per corpus)
    for corpus_name in os.listdir(base_dir):
        corpus_path = os.path.join(base_dir, corpus_name)
        es_dir = os.path.join(corpus_path, "es")
        en_dir = os.path.join(corpus_path, "en")

        if not (os.path.isdir(es_dir) and os.path.isdir(en_dir)):
            continue  # skip if the es/en structure is missing

        for fname in os.listdir(es_dir):
            if not fname.endswith("_es.txt"):
                continue
            if fname.startswith("RESP"):  # skip anomalous files
                continue

            base_name = fname.replace("_es.txt", "")
            es_file = os.path.join(es_dir, fname)
            en_file = os.path.join(en_dir, base_name + "_en.txt")

            if not os.path.exists(en_file):
                print(f"Pair not found for {fname} in {corpus_name}")
                continue

            # Load Spanish and English lines (keep blanks initially)
            with open(es_file, encoding="utf-8") as f_es:
                es_lines = [line.rstrip("\n\r") for line in f_es]
            with open(en_file, encoding="utf-8") as f_en:
                en_lines = [line.rstrip("\n\r") for line in f_en]

            # Remove trailing empty lines
            while es_lines and not es_lines[-1].strip():
                es_lines.pop()
            while en_lines and not en_lines[-1].strip():
                en_lines.pop()

            # Remove internal empty lines
            es_lines = [line.strip() for line in es_lines if line.strip()]
            en_lines = [line.strip() for line in en_lines if line.strip()]

            # Check alignment
            if len(es_lines) != len(en_lines):
                diff = abs(len(es_lines) - len(en_lines))
                info = {
                    "corpus": corpus_name,
                    "filename": base_name,
                    "es_lines": len(es_lines),
                    "en_lines": len(en_lines),
                    "diff": diff,
                }
                if diff <= 3:
                    reviewable.append(info)  # small misalignments to review
                else:
                    non_reviewable.append(info)  # discard larger misalignments
                continue

            # Add aligned sentence pairs
            for es_line, en_line in zip(es_lines, en_lines):
                records.append(
                    {
                        "corpus": corpus_name,
                        "filename": base_name,
                        "es": es_line,
                        "en": en_line,
                    }
                )

    # Build final DataFrames
    parallel_df = pd.DataFrame(records)
    reviewable_df = pd.DataFrame(reviewable)
    non_reviewable_df = pd.DataFrame(non_reviewable)

    return parallel_df, reviewable_df, non_reviewable_df

In [4]:
def tokenize(text: str) -> List[str]:
    """
    Tokenize a string into lowercase word tokens using regex.

    Args:
        text: Input string.

    Returns:
        A list of tokens (words).
    """
    return re.findall(r"\b\w+\b", str(text).lower())

In [5]:
parallel_df, reviewable_df, non_reviewable_df = build_parallel_dataframe()

In [6]:
parallel_df.head()

,corpus,filename,es,en
0,FP_FOLLETOS,KP_PL KP CONSUMER,SIN GLUTEN,GLUTEN FREE
1,FP_FOLLETOS,KP_PL KP CONSUMER,SIN LACTOSA,LACTOSE FREE
2,FP_FOLLETOS,KP_PL KP CONSUMER,SIN SACAROSA,SUCROSE FREE
3,FP_FOLLETOS,KP_PL KP CONSUMER,PRINCIPIO ACTIVO:,ACTIVE INGREDIENTS:
4,FP_FOLLETOS,KP_PL KP CONSUMER,Ibuprofeno.,Ibuprofen.


In [7]:
parallel_df.shape

(32064, 4)

In [8]:
parallel_df = parallel_df.drop_duplicates()

In [9]:
parallel_df.shape

(31813, 4)

In [10]:
parallel_df.head()

,corpus,filename,es,en
0,FP_FOLLETOS,KP_PL KP CONSUMER,SIN GLUTEN,GLUTEN FREE
1,FP_FOLLETOS,KP_PL KP CONSUMER,SIN LACTOSA,LACTOSE FREE
2,FP_FOLLETOS,KP_PL KP CONSUMER,SIN SACAROSA,SUCROSE FREE
3,FP_FOLLETOS,KP_PL KP CONSUMER,PRINCIPIO ACTIVO:,ACTIVE INGREDIENTS:
4,FP_FOLLETOS,KP_PL KP CONSUMER,Ibuprofeno.,Ibuprofen.


In [13]:
reviewable_df.head()

,corpus,filename,es_lines,en_lines,diff
0,FP_FOLLETOS,KP_PL GYNEA,664,666,2
1,FARMA_PARALELO,MDL00144,36,35,1
2,FARMA_PARALELO,MDL01718,11,10,1
3,FARMA_PARALELO,MDL01452,67,68,1
4,FARMA_PARALELO,MDL00365,8,7,1


In [14]:
reviewable_df.shape

(229, 5)

In [15]:
non_reviewable_df.head()

,corpus,filename,es_lines,en_lines,diff
0,FARMA_PARALELO,MDL00475,4,9,5
1,FARMA_PARALELO,MDL01620,107,39,68
2,FARMA_PARALELO,MDL00745,47,21,26
3,FARMA_PARALELO,MDL01390,52,4,48
4,FARMA_PARALELO,MDL01715,7,11,4


In [16]:
non_reviewable_df.shape

(75, 5)

In [ ]:
# Create output directory if it does not exist
os.makedirs(OUT_DIR, exist_ok=True)

groups = parallel_df["filename"]   # grouping by document
y = parallel_df["corpus"]          # stratification variable

# Outer stratified group k-fold
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (trainval_idx, test_idx) in enumerate(sgkf.split(parallel_df, y, groups)):

    # Split into train+val and test
    trainval_df = parallel_df.iloc[trainval_idx]
    groups_tv = trainval_df["filename"]
    y_tv = trainval_df["corpus"]

    # Inner split for validation (80/20 ratio)
    sgkf_inner = StratifiedGroupKFold(
        n_splits=int(1 / 0.2),  # 5 splits → 80% train / 20% val
        shuffle=True,
        random_state=SEED + fold
    )
    inner_splits = list(sgkf_inner.split(trainval_df, y_tv, groups_tv))

    inner_train_idx, val_idx = inner_splits[0]

    # Extract indices for each split
    train_idx = trainval_df.iloc[inner_train_idx].index
    val_idx = trainval_df.iloc[val_idx].index
    test_idx = parallel_df.iloc[test_idx].index

    # Build final dataframes
    df_train = parallel_df.loc[train_idx]
    df_val = parallel_df.loc[val_idx]
    df_test = parallel_df.loc[test_idx]

    # Save fold to disk
    fold_dir = os.path.join(OUT_DIR, f"fold_{fold+1}")
    os.makedirs(fold_dir, exist_ok=True)

    df_train.to_csv(os.path.join(fold_dir, "train.tsv"), sep="\t", index=False)
    df_val.to_csv(os.path.join(fold_dir, "val.tsv"), sep="\t", index=False)
    df_test.to_csv(os.path.join(fold_dir, "test.tsv"), sep="\t", index=False)